# Midsem Question 3 — TF-IDF on Toy Dataset

**Sentences (no start/end markers):**
- S1: `cat dog animal`
- S2: `cat cat pet`
- S3: `dog pet animal`
- S4: `cat dog pet`

**Settings:** TF = raw counts; IDF = unsmoothed `log(N / df(t))` (natural log); N = 4 sentences.

**Goal:** TF-IDF vector per sentence + find the two closest sentences (cosine similarity).

In [1]:
import math
import numpy as np

sentences = [
    "cat dog animal",
    "cat cat pet",
    "dog pet animal",
    "cat dog pet",
]

                         
vocab = sorted({w for s in sentences for w in s.split()})
print("Vocabulary:", vocab)

N = len(sentences)
tf = {}
df = {w: 0 for w in vocab}
for i, s in enumerate(sentences):
    words = s.split()
    counts = {w: words.count(w) for w in vocab}
    tf[i] = counts
    for w in vocab:
        if counts[w] > 0:
            df[w] += 1

print("\nDF (document frequency):")
for w in vocab:
    print(f"  {w:8s} df={df[w]}")

                          
idf = {w: math.log(N / df[w]) for w in vocab}
print("\nIDF = log(N/df):")
for w in vocab:
    print(f"  {w:8s} idf = log({N}/{df[w]}) = {idf[w]:.6f}")

                        
tfidf = {}
for i in range(N):
    tfidf[i] = np.array([tf[i][w] * idf[w] for w in vocab], dtype=float)

print("\nTF-IDF vectors (order: cat, dog, animal, pet):")
for i in range(N):
    print(f"  S{i+1}: {tfidf[i].round(6)}")

Vocabulary: ['animal', 'cat', 'dog', 'pet']

DF (document frequency):
  animal   df=2
  cat      df=3
  dog      df=3
  pet      df=3

IDF = log(N/df):
  animal   idf = log(4/2) = 0.693147
  cat      idf = log(4/3) = 0.287682
  dog      idf = log(4/3) = 0.287682
  pet      idf = log(4/3) = 0.287682

TF-IDF vectors (order: cat, dog, animal, pet):
  S1: [0.693147 0.287682 0.287682 0.      ]
  S2: [0.       0.575364 0.       0.287682]
  S3: [0.693147 0.       0.287682 0.287682]
  S4: [0.       0.287682 0.287682 0.287682]


In [2]:
                                                   
def cosine(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

print("Cosine similarity matrix:")
print("      " + " ".join(f"S{j+1:>2}" for j in range(N)))
sims = []
for i in range(N):
    row = []
    for j in range(N):
        s = cosine(tfidf[i], tfidf[j])
        row.append(s)
        if i < j:
            sims.append((s, i + 1, j + 1))
    print(f"S{i+1} " + " ".join(f"{v:6.4f}" for v in row))

sims.sort(reverse=True)
print("\nClosest sentence pair:")
for s, i, j in sims:
    print(f"  S{i} <-> S{j}: similarity = {s:.6f}")
best = sims[0]
print(f"\n=> The two closest sentences are S{best[1]} and S{best[2]} (similarity {best[0]:.6f}).")

Cosine similarity matrix:
      S 1 S 2 S 3 S 4
S1 1.0000 0.3201 0.8719 0.4133
S2 0.3201 1.0000 0.1601 0.7746
S3 0.8719 0.1601 1.0000 0.4133
S4 0.4133 0.7746 0.4133 1.0000

Closest sentence pair:
  S1 <-> S3: similarity = 0.871882
  S2 <-> S4: similarity = 0.774597
  S3 <-> S4: similarity = 0.413308
  S1 <-> S4: similarity = 0.413308
  S1 <-> S2: similarity = 0.320147
  S2 <-> S3: similarity = 0.160074

=> The two closest sentences are S1 and S3 (similarity 0.871882).


## Answer

Using raw-count TF and unsmoothed IDF (`log(4/df)`), the TF-IDF vectors are:

| Term | idf | S1 | S2 | S3 | S4 |
|------|-----|-----|-----|-----|-----|
| cat | log(4/3)≈0.2877 | 0.2877 | 0.5754 | 0 | 0.2877 |
| dog | log(4/3)≈0.2877 | 0.2877 | 0 | 0.2877 | 0.2877 |
| animal | log(4/2)≈0.6931 | 0.6931 | 0 | 0.6931 | 0 |
| pet | log(4/3)≈0.2877 | 0 | 0.2877 | 0.2877 | 0.2877 |

Cosine similarities show the maximum between **S1 (`cat dog animal`)** and **S3 (`dog pet animal`)** — they share the content words `dog` and `animal` (including the higher-IDF term `animal`).

**Closest pair: S1 and S3** (cosine similarity ≈ 0.872).

> Note: changing the log base scales every IDF by the same constant, which scales every vector equally and leaves cosine-similarity rankings unchanged, so the closest pair is invariant to the log base.